# Ordered Logistic Regression Results Dataset Exploration with `mlcroissant`
This notebook provides a complete guide to loading and exploring a dataset using the [`mlcroissant`](https://mlcroissant.readthedocs.io/en/latest/) library, following best practices for referencing all dataset components by their `@id` fields.

### Dataset Source
The dataset is described by a Croissant schema available at:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

The data covers ordered logistic regression outputs for adoption predictors of indigenous and modern knowledge in rangeland management practices across Northern Kenya.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`. The metadata will offer a summary of the dataset, including name and description.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset from Croissant schema
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset loaded: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview

Explore the Croissant schema for available record sets (tables/files), their fields, and the `@id` of each entity. `mlcroissant` represents record sets and fields as objects with attributes, and `@id` uniquely identifies each element for downstream access.

In [ ]:
# List all record sets with their @id
record_sets = list(dataset.record_sets)

print("Available record sets (tables/files):")
for rs in record_sets:
    print(f"  - Name: {rs.name}, @id: {rs.id}")

# For each record set, list the fields and columns with their @id
recordset_fields = {}
for rs in record_sets:
    print(f"\nRecord Set: {rs.name} (@id: {rs.id})")
    fields = list(rs.fields)
    columns = list(rs.columns)
    recordset_fields[rs.id] = [f.id for f in fields]
    if fields:
        print("  Fields:")
        for f in fields:
            print(f"    * {f.name} (@id: {f.id}, type: {f.data_type})")
    if columns:
        print("  Columns:")
        for col in columns:
            print(f"    * {col.name} (@id: {col.id}, type: {col.data_type})")

## 3. Data Extraction

Load data from each record set using their `@id` into a pandas DataFrame. Refer to the overview above for choosing which record sets and fields are relevant. Access columns exclusively via their `@id` to ensure programmatic correctness.

**Note:** Replace `<record_set_id>` with actual `@id` values based on your dataset (see overview cell above).

In [ ]:
# Prepare data extraction
# Select record sets to load (by @id): modify this list based on the overview above if desired
record_set_ids = [rs.id for rs in dataset.record_sets]

# Load data from each record set into a pandas DataFrame, using the @id as key
dfs = {}
for rs_id in record_set_ids:
    data = list(dataset.records(record_set=rs_id))
    if data:
        dfs[rs_id] = pd.DataFrame(data)
        print(f"Loaded {len(dfs[rs_id])} records for record set {rs_id}")
    else:
        print(f"No records found for record set {rs_id}")

# Display column names for one of the main record sets (replace with relevant @id)
if dfs:
    first_rs_id = list(dfs.keys())[0]
    print(f"\nFirst record set DataFrame columns (@id):")
    print(dfs[first_rs_id].columns.tolist())
    display(dfs[first_rs_id].head())

## 4. Exploratory Data Analysis (EDA)

Apply basic data processing: filter records, normalize numeric fields, group by categories. All columns are referenced strictly by their Croissant `@id` as established previously. Adjust the below to match actual `@id`s you wish to analyze from the data loaded.

In [ ]:
# Example: Select a record set with numeric fields for EDA
# Use the record set and field @ids found in the overview. Adjust as needed.

# Pick first non-empty DataFrame
record_set_id = first_rs_id
df = dfs[record_set_id]

# Identify a numeric field for demonstration (adjust @id as needed)
potential_numeric_columns = df.select_dtypes(include='number').columns.tolist()
# Use the first numeric column for demo, or override
if potential_numeric_columns:
    numeric_field_id = potential_numeric_columns[0]
else:
    print("No numeric fields detected in DataFrame.")

# Filter: Values above a threshold for demonstration
if potential_numeric_columns:
    threshold = df[numeric_field_id].mean()
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records where {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    )
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by another field if possible
    # Identify a non-numeric column for grouping
    other_columns = [col for col in df.columns if col != numeric_field_id]
    group_field_id = None
    for col in other_columns:
        if df[col].dtype == object:
            group_field_id = col
            break

    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())
    else:
        print("No suitable non-numeric field found for grouping.")

## 5. Visualization

Visualize distributions or relationships for the analyzed fields. Here, we show example histograms and scatterplots for the numeric columns. All axes and labels are referenced by their Croissant `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram for the selected numeric field
if potential_numeric_columns:
    plt.figure(figsize=(6,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    # If a group field exists, make a boxplot
    if group_field_id:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=30, ha='right')
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion

This notebook demonstrates how to load, inspect, and analyze a FAIR dataset described using the Croissant schema and accessed using `mlcroissant`. 

- **All entities (record sets, fields, columns)** have been referenced using their unique `@id` for clarity and reproducibility.
- Data selection, filtering, normalization, grouping, and basic visualizations were performed to showcase possible analyses on logistic regression and survey results.

`mlcroissant` offers a robust bridge between standards-based data packaging and analysis in Python, ensuring your workflows are portable and standards-compliant.